# 高级主题
## 限速
许多聊天服务提供商都会限制在给定时间内可以发起的请求次数。
如果达到速率限制，您通常会收到服务提供商的速率限制错误响应，需要等待一段时间才能再次发起请求。
为了帮助管理速率限制，聊天模型集成接受一个 rate_limiter 参数，该参数可以在初始化期间提供，以控制发出请求的速率。

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain.chat_models import init_chat_model

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # 1 request every 10s
    check_every_n_seconds=0.1,  # Check every 100ms whether allowed to make a request
    max_bucket_size=10,  # Controls the maximum burst size.
)

model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama",  # 想换OpenAI就改成 openai
    temperature=0,
    rate_limiter=rate_limiter  
)

## Token Usage
### 回调处理程序

In [ ]:
from langchain_core.callbacks import UsageMetadataCallbackHandler
callback = UsageMetadataCallbackHandler()

result_1 = model.invoke("Hello", config={"callbacks": [callback]})

print(callback.usage_metadata)

{'qwen2.5:3b-instruct-q4_K_M': {'input_tokens': 30, 'output_tokens': 29, 'total_tokens': 59}}


### 上下文管理器 with


In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import get_usage_metadata_callback

model_1 = init_chat_model(model="qwen2.5:3b-instruct-q4_K_M",model_provider="ollama",)

with get_usage_metadata_callback() as cb:
    model_1.invoke("Hello")
    print(cb.usage_metadata)

{'qwen2.5:3b-instruct-q4_K_M': {'input_tokens': 30, 'output_tokens': 41, 'total_tokens': 71}}


## 调用配置
调用模型时，可以通过config参数使用RunnableConfig字典传递额外的配置信息。这可以实现对执行行为、回调和元数据跟踪的运行时控制。
这些配置值在以下情况下尤其有用：
    使用LangSmith跟踪进行调试
    实现自定义日志记录或监控
    生产中资源利用的控制
    跟踪复杂管道中的调用

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import BaseCallbackHandler

# 自定义一个简单回调
class SimpleCallback(BaseCallbackHandler):
    def on_llm_start(self, *args, **kwargs):
        """
        名称固定，不能修改，模型开始推理时自动调用
        """
        print("✅ 模型开始运行")

    def on_llm_end(self, *args, **kwargs):
        """
        名称固定，不能修改，模型推理结束时自动调用
        """
        print("✅ 模型运行结束")

    # 模型报错
    def on_llm_error(self, error, *args, **kwargs):
        print(f"❌ 模型出错：{error}")

    # 流式输出（每个字都触发）
    def on_llm_new_token(self, token: str, *args, **kwargs):
        print(token, end="", flush=True)

# 初始化模型
model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama"
)

# 调用 + 完整 config
response = model.invoke(
    "Tell me a joke",
    config={
        "run_name": "joke_generation",
        "tags": ["humor", "demo"],
        "metadata": {"user_id": "123"},
        "callbacks": [SimpleCallback()]  # 👈 这里放回调
    }
)

print("\n👇 回答：")
print(response.content)

✅ 模型开始运行
Sure! Here's one for you:

Why did the tomato turn red?

Because it saw the salad dressing!

Hope that made your day a little brighter!✅ 模型运行结束

👇 回答：
Sure! Here's one for you:

Why did the tomato turn red?

Because it saw the salad dressing!

Hope that made your day a little brighter!


## 可配置模型


In [ ]:
from langchain.chat_models import init_chat_model

configurable_model = init_chat_model(temperature=0)

configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "qwen2.5:3b-instruct-q4_K_M", "model_provider":"ollama"}},
)

AIMessage(content='My name is Qwen.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b-instruct-q4_K_M', 'created_at': '2026-04-08T03:09:42.8360963Z', 'done': True, 'done_reason': 'stop', 'total_duration': 286718800, 'load_duration': 110692500, 'prompt_eval_count': 33, 'prompt_eval_duration': 39683100, 'eval_count': 7, 'eval_duration': 120168800, 'logprobs': None, 'model_name': 'qwen2.5:3b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--019d6b11-3313-7ae0-a09c-98b16cd95df6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 7, 'total_tokens': 40})

### 使用可配置模型进行声明式编程

In [7]:
from pydantic import BaseModel, Field


class GetWeather(BaseModel):
    """Get the current weather in a given location"""

    location: str = Field(description="The city and state, e.g. San Francisco, CA")


class GetPopulation(BaseModel):
    """Get the current population in a given location"""

    location: str = Field(description="The city and state, e.g. San Francisco, CA")


model = init_chat_model(temperature=0)
model_with_tools = model.bind_tools([GetWeather, GetPopulation])

model_with_tools.invoke(
    "what's bigger in 2024 LA or NYC", config={"configurable": {"model": "qwen2.5:3b-instruct-q4_K_M", "model_provider":"ollama"}}
).tool_calls

[{'name': 'GetPopulation',
  'args': {'location': 'Los Angeles, CA'},
  'id': '6a65be48-d1ae-4da3-98dc-28fc98d94ba9',
  'type': 'tool_call'},
 {'name': 'GetPopulation',
  'args': {'location': 'New York City, NY'},
  'id': '6920408a-36b9-4b02-baaa-eca18983d884',
  'type': 'tool_call'}]